In [1]:
from scipy.stats import chi2_contingency
import numpy as np
import pandas as pd

In [2]:
#Statistical test to validate diferencial hypothesis: Chi Square Test
#Is there a difference between stabilizers/stabilizers and no effect?

#Remember that it is important: Where is my storage dir? Where is my result dir? How many folders do I have?
matrix_path= '/projectbig/jupyternotebook/jrivera/output/pocket_results/ligand_tables/sorted_matrix/matrix_sum_labels.csv'

matrix= pd.read_csv(matrix_path)

In [124]:
#Define your Chi square test
#For this you need to import ch2_contingency and also define your function
#This code was provided by Behnam

def chi2(n1, n2, thr=0, return_table=True):
    try:
        n1 = np.array(n1)
        n2 = np.array(n2)
        
        n1_pos = np.sum(n1 > thr)
        n1_zero = np.sum(n1 <= thr)
        
        n2_pos = np.sum(n2 > thr)
        n2_zero = np.sum(n2 <= thr)
    
        # Contingency table (2x2)
        table = np.array([[n1_pos, n1_zero],
                          [n2_pos, n2_zero]])
    
        # print(table)
        
        chi2, p, dof, expected = chi2_contingency(table)
        p = p.item()
    except:
        p=1


    #if return_table:
    #    return (p, table)
    #else:
    #    return p 
    return (table) if return_table else p
                   #if return_table= True, but already there



In [125]:
#Now you want to apply your test to your categoried
#1. Define the categories from your datta set
#2. Use your filtering function to select the desired class
#matrix['label'] == 'stabilizer'
#from the matrix column called "label", take only tthe ones that are equal to "stabilizer"
#matrix['label'].isin(['strong', 'weak'])
#isin function to check if it is in equal to any of them two categories

#3. Then give the input to your function def chi2(n1, n2, thr=0)
#In this case matrix.loc[], interaction
#we are looping over the interactions that we defined early to get the p value from each of them
#Finally, make the values a list to futhter work with them 

#4. Now that we defined our n1 and n2, we can apply the chi2 function 
#chi2(n_stab, n_no, thr=thr)

#thr=thr because we will check for different thresholds and then select the one with the minimum value
#for thr in range(30)
# np.min(p_stab) -> numpy minimum!

#Result with 3 nachkommastellen: round(p_destab, 3) or p_destab:.3f-> python!

In [133]:
#P-values
interaction_set = [ 'hydrophobic','pistack','pication_ligand_ring','pication_ligand_charge','halogen_ligand_donnor','hbond_ligand_acceptor','hbond_ligand_donnor',
    'waterbridge','ionic_ligand_positive','ionic_ligand_negative','metal']

result = []


for interaction in interaction_set:
    n_stab = matrix.loc[matrix['label'] == 'stabilizer', interaction]
    n_destab = matrix.loc[matrix['label'] == 'destabilizer', interaction]
    n_no = matrix.loc[matrix['label'].isin(['strong', 'weak']), interaction]
    
    n_stab= list(n_stab)
    n_destab= list(n_destab)
    n_no= list(n_no)

    #print(n_stab, n_destab, n_no)
    
    p_stab = []
    p_destab = []
    
    for thr in range(100):  
        p_stab.append(chi2(n_stab, n_no, thr=thr, return_table=False))
        p_destab.append(chi2(n_destab, n_no, thr=thr, return_table=False))

    p_stab = np.min(p_stab)        #Select mininal value
    
    p_destab = np.min(p_destab)

    p_stab = round(p_stab, 3)      #Round up to 3 decimals
    
    p_destab = round(p_destab, 3)
    
    #print(p_stab, p_destab)

    result.append({
        'interaction': interaction,
        'p_value: stabilizer': p_stab,
        'p_value: destabilizer': p_destab})

print(result)

[{'interaction': 'hydrophobic', 'p_value: stabilizer': np.float64(0.322), 'p_value: destabilizer': np.float64(0.214)}, {'interaction': 'pistack', 'p_value: stabilizer': np.float64(0.276), 'p_value: destabilizer': np.float64(0.079)}, {'interaction': 'pication_ligand_ring', 'p_value: stabilizer': np.float64(0.732), 'p_value: destabilizer': np.float64(0.312)}, {'interaction': 'pication_ligand_charge', 'p_value: stabilizer': np.float64(0.24), 'p_value: destabilizer': np.float64(0.71)}, {'interaction': 'halogen_ligand_donnor', 'p_value: stabilizer': np.float64(0.276), 'p_value: destabilizer': np.float64(0.079)}, {'interaction': 'hbond_ligand_acceptor', 'p_value: stabilizer': np.float64(0.276), 'p_value: destabilizer': np.float64(0.099)}, {'interaction': 'hbond_ligand_donnor', 'p_value: stabilizer': np.float64(0.152), 'p_value: destabilizer': np.float64(0.075)}, {'interaction': 'waterbridge', 'p_value: stabilizer': np.float64(0.018), 'p_value: destabilizer': np.float64(0.005)}, {'interaction

In [119]:
#Convert your result into a data frame and save it

result_table = pd.DataFrame(data=result)
result_table
result_table.to_csv('/projectbig/jupyternotebook/jrivera/output/pocket_results/ligand_tables/p_values_by_interaction_sum_poses.csv', index=False)

In [143]:
#Contingecy table
interaction_set = [ 'hydrophobic','pistack','pication_ligand_ring','pication_ligand_charge','halogen_ligand_donnor','hbond_ligand_acceptor','hbond_ligand_donnor',
    'waterbridge','ionic_ligand_positive','ionic_ligand_negative','metal']

result = []


for interaction in interaction_set:
    n_stab = matrix.loc[matrix['label'] == 'stabilizer', interaction]
    n_destab = matrix.loc[matrix['label'] == 'destabilizer', interaction]
    n_no = matrix.loc[matrix['label'].isin(['strong', 'weak']), interaction]
    
    n_stab= list(n_stab)
    n_destab= list(n_destab)
    n_no= list(n_no)

    #print(n_stab, n_destab, n_no)
    
    p_stab = []
    p_destab = []

    table_stab = []
    table_destab = []
    
    for thr in range(100):  
        p_stab.append(chi2(n_stab, n_no, thr=thr, return_table=False))
        p_destab.append(chi2(n_destab, n_no, thr=thr, return_table=False))

        table_stab.append(chi2(n_stab, n_no, thr=thr, return_table=True))
        table_destab.append(chi2(n_destab, n_no, thr=thr, return_table=True))

    # Find the index of the lowest p-value
    min_idx = np.argmin(p_stab)                               #np.armin means return the index of the lowest value
    print("Lowest p-value:", p_stab[min_idx])                 #Gives the p_value at the lowest indexing
    print("Threshold:", min_idx)
    print("Contingency table:\n", table_stab[min_idx])        #\n is a newline character, so it moves the cursor to the next line after printing
                                                              #has nothing to do with fcuntion, just for output format

Lowest p-value: 0.32218881313557746
Threshold: 44
Contingency table:
 [[  9  17]
 [ 40 131]]
Lowest p-value: 0.2756779240549242
Threshold: 0
Contingency table:
 [[ 25   1]
 [171   0]]
Lowest p-value: 0.7324269667296892
Threshold: 0
Contingency table:
 [[  4  22]
 [ 35 136]]
Lowest p-value: 0.24024612965656084
Threshold: 0
Contingency table:
 [[  0  26]
 [ 15 156]]
Lowest p-value: 0.2756779240549242
Threshold: 6
Contingency table:
 [[  1  25]
 [  0 171]]
Lowest p-value: 0.2756779240549242
Threshold: 26
Contingency table:
 [[  1  25]
 [  0 171]]
Lowest p-value: 0.15227118579535073
Threshold: 8
Contingency table:
 [[  0  26]
 [ 19 152]]
Lowest p-value: 0.01823950984956973
Threshold: 12
Contingency table:
 [[  5  21]
 [  8 163]]
Lowest p-value: 0.21168861627631705
Threshold: 0
Contingency table:
 [[ 13  13]
 [ 60 111]]
Lowest p-value: 1.0
Threshold: 0
Contingency table:
 [[  7  19]
 [ 48 123]]
Lowest p-value: 1
Threshold: 0
Contingency table:
 [[  0  26]
 [  0 171]]


In [27]:
try:
    a = 1 + '3'
except:
    a = 0
a

0

In [139]:
a = True


a == True

True

In [140]:
if a:
    print('hiii')

hiii


In [141]:
b = False


b == True

False

In [142]:
if b:
    print('bye')